# Transformers para la clasificación de emociones

Curso: Minería de Datos

Proyecto: Clasificación automática de emociones en publicaciones de jóvenes peruanos

### 1. Preparación del entorno

#### Verificar la GPU

Utilizaremos Google Colab para aprovechar su GPU y que el proceso sea mucho más rápido.

In [45]:
import torch
import sys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo detectado: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[!] NO se detectó GPU. El entrenamiento será muy lento.")

Dispositivo detectado: cuda
GPU: Tesla T4
Memoria: 15.6 GB


#### Entorno de ejecución

In [46]:
import os
import sys

if os.path.exists('/content'):
    # Entorno de la nube
    print("Entorno de la nube (Colab) detectado.")
    print("Montando Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    BASE_DIR = '/content'
    # Ruta a la carpeta donde se encuentra los datasets limpios
    DATOS_DIR = '/content/drive/MyDrive/Proyecto-Mineria-G4/prueba'
else:
    # Entorno local
    print("Entorno local detectado.")
    try:
        current_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        current_dir = os.getcwd()

    BASE_DIR = current_dir
    while BASE_DIR != os.path.dirname(BASE_DIR):
        if 'modelado' in os.listdir(BASE_DIR) and os.path.isdir(os.path.join(BASE_DIR, 'modelado')):
            break
        BASE_DIR = os.path.dirname(BASE_DIR)
        
    DATOS_DIR = os.path.join(BASE_DIR, 'modelado', 'datos')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

RESULTADOS_DIR = os.path.join(BASE_DIR, 'resultados_transformers')
os.makedirs(RESULTADOS_DIR, exist_ok=True)

print(f"Directorio base detectado: {BASE_DIR}")
print(f"Directorio de datos: {DATOS_DIR}")


Entorno de la nube (Colab) detectado.
Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio base detectado: /content
Directorio de datos: /content/drive/MyDrive/Proyecto-Mineria-G4/prueba


#### Instalación de dependencias

In [47]:
!pip install transformers datasets pysentimiento evaluate accelerate scikit-learn pandas matplotlib seaborn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 20.5 MB/s eta 0:00:00


#### Importación de librerías

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from sklearn.preprocessing import LabelBinarizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
    EarlyStoppingCallback
)
from accelerate import Accelerator

print("Todas las librerías se importaron correctamente.")

Todas las librerías se importaron correctamente.


### 2. Carga y estandarización de los datasets

Se cargan los 3 datasets preprocesados (YouTube parte 1, YouTube parte 2, TikTok) y se unifican en un solo dataframe.

In [49]:
import glob
import pandas as pd

archivos = [
    os.path.join(DATOS_DIR, 'dataset_procesado_tiktok_parte01.csv'),
    os.path.join(DATOS_DIR, 'dataset_procesado_youtube_parte01.csv'),
    os.path.join(DATOS_DIR, 'dataset_procesado_youtube_parte02.csv')
]

dfs = []
for f in archivos:
    if os.path.exists(f):
        print(f"Cargando {os.path.basename(f)}...")
        temp_df = pd.read_csv(f, sep=';', encoding='utf-8')
        temp_df.columns = temp_df.columns.str.strip().str.replace('"', '').str.lower()
        dfs.append(temp_df)

if not dfs:
    raise ValueError("No se encontraron los datasets en modelado/datos/")

df = pd.concat(dfs, ignore_index=True)
df['emocion'] = df['emocion'].astype(str).str.strip().str.replace('"', '').str.capitalize()
df['emocion'] = df['emocion'].replace({'Alegría': 'Alegria'})

# Filtrar 6 emociones maestras
emociones_validas = ['Sorpresa', 'Miedo', 'Alegria', 'Tristeza']
df = df[df['emocion'].isin(emociones_validas)]
# Usamos la nueva columna del dataset consolidado
df = df.dropna(subset=['content_transformers', 'emocion'])

print(f"\nDataset unificado: {len(df)} registros")
print(f"Distribución de emociones:\n{df['emocion'].value_counts()}")


Cargando dataset_procesado_tiktok_parte01.csv...
Cargando dataset_procesado_youtube_parte01.csv...
Cargando dataset_procesado_youtube_parte02.csv...

Dataset unificado: 9065 registros
Distribución de emociones:
emocion
Tristeza    2684
Alegria     2329
Sorpresa    2048
Miedo       2004
Name: count, dtype: int64


### 3. División y preparación del conjunto de datos

In [50]:
# Se utiliza la columna content_transformers
textos = df['content_transformers'].astype(str).tolist()
etiquetas = df['emocion'].tolist()
clases = sorted(df['emocion'].unique().tolist())
print(f"Clases: {clases}")

# División del conjunto de los datos: 70% de entrenamiento y 30% de prueba
train_texts, test_texts, train_labels, test_labels = train_test_split(
    textos, etiquetas, test_size=0.3, random_state=42, stratify=etiquetas
)

print(f"\nEntrenamiento: {len(train_texts)} registros")
print(f"Prueba: {len(test_texts)} registros")

# Mapa de etiquetas a índices
label2id = {e: i for i, e in enumerate(clases)}
id2label = {i: e for i, e in enumerate(clases)}
print(f"\nMapa de etiquetas: {label2id}")

train_labels_ids = [label2id[l] for l in train_labels]
test_labels_ids = [label2id[l] for l in test_labels]

Clases: ['Alegria', 'Miedo', 'Sorpresa', 'Tristeza']

Entrenamiento: 6345 registros
Prueba: 2720 registros

Mapa de etiquetas: {'Alegria': 0, 'Miedo': 1, 'Sorpresa': 2, 'Tristeza': 3}


#### Pesos por clase (para manejar desbalance)

In [51]:
from sklearn.utils.class_weight import compute_class_weight

classes_array = np.array(clases)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes_array,
    y=np.array(train_labels)
)
class_weights_dict = {clases[i]: class_weights[i] for i in range(len(clases))}
pesos_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Pesos por clase (balanced):")
for c, w in class_weights_dict.items():
    print(f"   {c}: {w:.4f}")

Pesos por clase (balanced):
   Alegria: 0.9732
   Miedo: 1.1306
   Sorpresa: 1.1069
   Tristeza: 0.8442


#### Clase Dataset de PyTorch

In [52]:
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

## 4. Funciones de evaluación

Se definen las funciones para calcular métricas y generar los gráficos (matriz de confusión y curva ROC multiclase).

In [53]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision_macro': precision_score(labels, predictions, average='macro', zero_division=0),
        'recall_macro': recall_score(labels, predictions, average='macro', zero_division=0),
        'f1_macro': f1_score(labels, predictions, average='macro', zero_division=0)
    }


def generar_matriz_confusion(y_true, y_pred, clases, titulo, filename, ruta):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(clases)))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=clases, yticklabels=clases)
    plt.title(f'Matriz de Confusión - {titulo}')
    plt.xlabel('Predicción')
    plt.ylabel('Real')
    plt.xticks(rotation=45)
    plt.tight_layout()
    ruta_completa = os.path.join(ruta, filename)
    plt.savefig(ruta_completa, dpi=300)
    plt.close()
    return ruta_completa


def generar_curva_roc(y_true, y_scores, clases, titulo, filename, ruta):
    lb = LabelBinarizer()
    y_true_bin = lb.fit_transform(y_true)

    plt.figure(figsize=(10, 8))

    fpr_dict = {}
    tpr_dict = {}
    roc_auc = {}

    for i, clase in enumerate(clases):
        fpr_dict[i], tpr_dict[i], _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
        roc_auc[i] = auc(fpr_dict[i], tpr_dict[i])
        plt.plot(fpr_dict[i], tpr_dict[i], lw=2,
                 label=f'{clase} (AUC = {roc_auc[i]:.3f})')

    # Macro-average
    all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(len(clases))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(clases)):
        mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
    mean_tpr /= len(clases)
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, 'k--', lw=3,
             label=f'Macro-promedio (AUC = {macro_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'gray', linestyle=':', lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos (FPR)')
    plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
    plt.title(f'Curva ROC - {titulo}')
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    ruta_completa = os.path.join(ruta, filename)
    plt.savefig(ruta_completa, dpi=300)
    plt.close()
    return ruta_completa, roc_auc, macro_auc


def reporte_por_clase(y_true, y_pred, clases):
    report = classification_report(y_true, y_pred, labels=range(len(clases)),
                                   target_names=clases, zero_division=0, output_dict=True)
    return report


def guardar_reporte_markdown(filename, ruta, nombre_modelo, metricas, cm_path, roc_path,
                             roc_auc_dict, macro_auc, reporte_clases, y_true, y_pred, clases):
    ruta_completa = os.path.join(ruta, filename)
    cm_rel = cm_path.replace('\\', '/')
    roc_rel = roc_path.replace('\\', '/')

    with open(ruta_completa, 'w', encoding='utf-8') as f:
        f.write(f"# Reporte de Evaluación - {nombre_modelo}\n\n")
        f.write(f"Resultados de métricas de rendimiento para el modelo {nombre_modelo}.\n\n")

        f.write("## Métricas Globales\n\n")
        f.write(f"- **Accuracy:** {metricas['accuracy']:.4f}\n")
        f.write(f"- **Precision (macro):** {metricas['precision_macro']:.4f}\n")
        f.write(f"- **Recall (macro):** {metricas['recall_macro']:.4f}\n")
        f.write(f"- **F1-Score (macro):** {metricas['f1_macro']:.4f}\n\n")

        f.write("## AUC (Área Bajo la Curva ROC)\n\n")
        for i, clase in enumerate(clases):
            f.write(f"- **{clase}:** {roc_auc_dict[i]:.4f}\n")
        f.write(f"- **Macro-promedio:** {macro_auc:.4f}\n\n")

        f.write("## Reporte por Clase\n\n")
        f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
        f.write("|-------|-----------|--------|----------|---------|\n")
        for clase in clases:
            r = reporte_clases[clase]
            f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
        f.write("\n")

        f.write("## Matriz de Confusión\n\n")
        f.write(f"![Matriz de Confusión {nombre_modelo}]({cm_rel})\n\n")

        f.write("## Curva ROC\n\n")
        f.write(f"![Curva ROC {nombre_modelo}]({roc_rel})\n\n")

        f.write("## Classification Report (detallado)\n\n")
        f.write("```text\n")
        f.write(classification_report(y_true, y_pred, labels=range(len(clases)),
                                      target_names=clases, zero_division=0))
        f.write("```\n")

    print(f"Reporte guardado: {ruta_completa}")
    return ruta_completa

#### Hiperparametros compartidos

In [54]:
MAX_LENGTH   = 128
BATCH_SIZE   = 16
EPOCHS       = 5       # early stopping controla el límite real
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200

### 5. Entrenamiento: BETO (BERT para corpus en español)

Modelo: [`dccuchile/bert-base-spanish-wwm-cased`](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased)

In [55]:
MODELO_BETO = 'dccuchile/bert-base-spanish-wwm-cased'
LEARNING_RATE_BETO = 3e-5

print(f"Configuración del Modelo Base:")
print(f"   Modelo: {MODELO_BETO}")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Warmup steps: {WARMUP_STEPS}")

Configuración del Modelo Base:
   Modelo: dccuchile/bert-base-spanish-wwm-cased
   Max length: 128
   Batch size: 16
   Epochs: 5
   Learning rate: 2e-05
   Weight decay: 0.01
   Warmup steps: 200


In [56]:
print("Cargando tokenizer...")
tokenizer_beto = AutoTokenizer.from_pretrained(MODELO_BETO)

print("Tokenizando textos de entrenamiento...")
train_encodings_beto = tokenizer_beto(
    train_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

print("Tokenizando textos de prueba...")
test_encodings_beto = tokenizer_beto(
    test_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

train_dataset_beto = EmotionDataset(train_encodings_beto, train_labels_ids)
test_dataset_beto = EmotionDataset(test_encodings_beto, test_labels_ids)

print(f"Train dataset: {len(train_dataset_beto)} muestras")
print(f"Test dataset: {len(test_dataset_beto)} muestras")

Cargando tokenizer...
Tokenizando textos de entrenamiento...
Tokenizando textos de prueba...
Train dataset: 6345 muestras
Test dataset: 2720 muestras


#### Función para entrenar los modelos

In [57]:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=pesos_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

print("CustomTrainer implentada.")

CustomTrainer implentada.


#### Entrenamiento del modelo

In [58]:
print("Cargando modelo pre-entrenado...")
model_beto = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BETO,
    num_labels=len(clases),
    id2label=id2label,
    label2id=label2id,
).to(device)

training_args_beto = TrainingArguments(
    output_dir='./resultados_beto',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_dir='./logs_beto',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    learning_rate=LEARNING_RATE_BETO,
)

trainer_beto = CustomTrainer(
    model=model_beto,
    args=training_args_beto,
    train_dataset=train_dataset_beto,
    eval_dataset=test_dataset_beto,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Iniciando entrenamiento...")
trainer_beto.train()

Cargando modelo pre-entrenado...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Iniciando entrenamiento...


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.885906,0.862960,0.669485,0.712019,0.666464,0.668092
2,0.605292,0.750046,0.735294,0.744667,0.735307,0.734978
3,0.326653,0.886575,0.712500,0.715246,0.713886,0.712780
4,0.213215,1.145759,0.726838,0.724868,0.728481,0.726194


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1588, training_loss=0.5460611266513316, metrics={'train_runtime': 708.7447, 'train_samples_per_second': 44.762, 'train_steps_per_second': 2.801, 'total_flos': 1669469624709120.0, 'train_loss': 0.5460611266513316, 'epoch': 4.0})

#### Evaluación del modelo

In [59]:
print("Evaluando BETO en conjunto de prueba...")
beto_preds = trainer_beto.predict(test_dataset_beto)
beto_logits = beto_preds.predictions
beto_preds_ids = np.argmax(beto_logits, axis=-1)

# Métricas globales
beto_metrics = {
    'accuracy': accuracy_score(test_labels_ids, beto_preds_ids),
    'precision_macro': precision_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0),
    'recall_macro': recall_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0),
    'f1_macro': f1_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0)
}

print(f"\nResultados BETO - Test:")
print(f"   Accuracy: {beto_metrics['accuracy']:.4f}")
print(f"   Precision (macro): {beto_metrics['precision_macro']:.4f}")
print(f"   Recall (macro): {beto_metrics['recall_macro']:.4f}")
print(f"   F1-Score (macro): {beto_metrics['f1_macro']:.4f}")

# Matriz de confusión
beto_cm_path = generar_matriz_confusion(
    test_labels_ids, beto_preds_ids, clases,
    'BETO', 'cm_beto.png', RESULTADOS_DIR
)
print(f"Matriz de confusión guardada: {beto_cm_path}")

# Reporte por clase
beto_report = reporte_por_clase(test_labels_ids, beto_preds_ids, clases)

# Curva ROC
beto_probas = torch.nn.functional.softmax(torch.tensor(beto_logits), dim=-1).numpy()
beto_roc_path, beto_auc_dict, beto_macro_auc = generar_curva_roc(
    test_labels_ids, beto_probas, clases,
    'BETO', 'roc_beto.png', RESULTADOS_DIR
)
print(f"Curva ROC guardada: {beto_roc_path}")

print(f"\nAUC por clase:")
for i, clase in enumerate(clases):
    print(f"   {clase}: {beto_auc_dict[i]:.4f}")
print(f"   Macro-promedio: {beto_macro_auc:.4f}")

Evaluando BETO en conjunto de prueba...



Resultados BETO - Test:
   Accuracy: 0.7353
   Precision (macro): 0.7447
   Recall (macro): 0.7353
   F1-Score (macro): 0.7350
Matriz de confusión guardada: /content/resultados_transformers/cm_beto.png
Curva ROC guardada: /content/resultados_transformers/roc_beto.png

AUC por clase:
   Alegria: 0.8766
   Miedo: 0.9488
   Sorpresa: 0.9030
   Tristeza: 0.9154
   Macro-promedio: 0.9111


### 6. Entrenamiento: RoBERTuito (RoBERTa en español)

Modelo: [`pysentimiento/robertuito-base-uncased`](https://huggingface.co/pysentimiento/robertuito-base-uncased)

In [60]:
from pysentimiento.preprocessing import preprocess_tweet

MODELO_ROBERTA = 'pysentimiento/robertuito-base-uncased'
LEARNING_RATE_ROBERTA = 1e-5

print(f"Configuración del Modelo Base:")
print(f"   Modelo: {MODELO_ROBERTA}")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Warmup steps: {WARMUP_STEPS}")

Configuración del Modelo Base:
   Modelo: pysentimiento/robertuito-base-uncased
   Max length: 128
   Batch size: 16
   Epochs: 5
   Learning rate: 2e-05
   Weight decay: 0.01
   Warmup steps: 200


In [61]:
print("Cargando tokenizer RoBERTa...")
tokenizer_roberta = AutoTokenizer.from_pretrained('pysentimiento/robertuito-base-uncased')

train_texts_proc = [preprocess_tweet(t) for t in train_texts]
test_texts_proc  = [preprocess_tweet(t) for t in test_texts]

print("Tokenizando textos de entrenamiento...")
train_encodings_roberta = tokenizer_roberta(
    train_texts_proc,   # textos preprocesados
    truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors=None
)

print("Tokenizando textos de prueba...")
test_encodings_roberta = tokenizer_roberta(
    test_texts_proc,    # textos preprocesados
    truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors=None
)

train_dataset_roberta = EmotionDataset(train_encodings_roberta, train_labels_ids)
test_dataset_roberta = EmotionDataset(test_encodings_roberta, test_labels_ids)

print(f"Train dataset: {len(train_dataset_roberta)} muestras")
print(f"Test dataset: {len(test_dataset_roberta)} muestras")


Cargando tokenizer RoBERTa...


config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/323 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/858k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Tokenizando textos de entrenamiento...
Tokenizando textos de prueba...
Train dataset: 6345 muestras
Test dataset: 2720 muestras


In [62]:
print("Cargando modelo RoBERTa...")
model_roberta = AutoModelForSequenceClassification.from_pretrained(
    MODELO_ROBERTA,              
    num_labels=len(clases),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)


training_args_roberta = TrainingArguments(
    output_dir='/content/checkpoints_roberta',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE_ROBERTA,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_dir='/content/logs_roberta',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    save_total_limit=1,
)

trainer_roberta = CustomTrainer(
    model=model_roberta,
    args=training_args_roberta,
    train_dataset=train_dataset_roberta,
    eval_dataset=test_dataset_roberta,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Iniciando entrenamiento de RoBERTa...")
trainer_roberta.train()
print("Entrenamiento de RoBERTa completado.")



Cargando modelo RoBERTa...


model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Iniciando entrenamiento de RoBERTa...
{'loss': '1.4', 'grad_norm': '5.51', 'learning_rate': '2.45e-06', 'epoch': '0.1259'}
{'loss': '1.355', 'grad_norm': '5.506', 'learning_rate': '4.95e-06', 'epoch': '0.2519'}
{'loss': '1.301', 'grad_norm': '5.846', 'learning_rate': '7.45e-06', 'epoch': '0.3778'}
{'loss': '1.192', 'grad_norm': '5.918', 'learning_rate': '9.95e-06', 'epoch': '0.5038'}
{'loss': '1.054', 'grad_norm': '5.688', 'learning_rate': '9.725e-06', 'epoch': '0.6297'}
{'loss': '1.017', 'grad_norm': '7.709', 'learning_rate': '9.445e-06', 'epoch': '0.7557'}
{'loss': '0.9768', 'grad_norm': '7.541', 'learning_rate': '9.165e-06', 'epoch': '0.8816'}
{'eval_loss': '0.8417', 'eval_accuracy': '0.6801', 'eval_precision_macro': '0.6806', 'eval_recall_macro': '0.6839', 'eval_f1_macro': '0.6805', 'eval_runtime': '5.048', 'eval_samples_per_second': '538.8', 'eval_steps_per_second': '33.68', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

{'loss': '0.6359', 'grad_norm': '10.03', 'learning_rate': '4.403e-06', 'epoch': '3.023'}
{'loss': '0.5984', 'grad_norm': '8.791', 'learning_rate': '4.123e-06', 'epoch': '3.149'}
{'loss': '0.592', 'grad_norm': '8.024', 'learning_rate': '3.843e-06', 'epoch': '3.275'}
{'loss': '0.565', 'grad_norm': '6.042', 'learning_rate': '3.563e-06', 'epoch': '3.401'}
{'loss': '0.5846', 'grad_norm': '8.391', 'learning_rate': '3.283e-06', 'epoch': '3.526'}
{'loss': '0.565', 'grad_norm': '13.62', 'learning_rate': '3.003e-06', 'epoch': '3.652'}
{'loss': '0.5653', 'grad_norm': '12.1', 'learning_rate': '2.723e-06', 'epoch': '3.778'}
{'loss': '0.5647', 'grad_norm': '8.912', 'learning_rate': '2.443e-06', 'epoch': '3.904'}
{'eval_loss': '0.7508', 'eval_accuracy': '0.7243', 'eval_precision_macro': '0.7248', 'eval_recall_macro': '0.7244', 'eval_f1_macro': '0.7238', 'eval_runtime': '5.014', 'eval_samples_per_second': '542.5', 'eval_steps_per_second': '33.91', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.5406', 'grad_norm': '8.114', 'learning_rate': '2.162e-06', 'epoch': '4.03'}
{'loss': '0.5472', 'grad_norm': '14.54', 'learning_rate': '1.882e-06', 'epoch': '4.156'}
{'loss': '0.5258', 'grad_norm': '11.14', 'learning_rate': '1.602e-06', 'epoch': '4.282'}
{'loss': '0.518', 'grad_norm': '8.044', 'learning_rate': '1.322e-06', 'epoch': '4.408'}
{'loss': '0.5036', 'grad_norm': '10.69', 'learning_rate': '1.042e-06', 'epoch': '4.534'}
{'loss': '0.5054', 'grad_norm': '12.08', 'learning_rate': '7.619e-07', 'epoch': '4.66'}
{'loss': '0.5169', 'grad_norm': '7.444', 'learning_rate': '4.818e-07', 'epoch': '4.786'}
{'loss': '0.5463', 'grad_norm': '8.319', 'learning_rate': '2.017e-07', 'epoch': '4.912'}
{'eval_loss': '0.7607', 'eval_accuracy': '0.7239', 'eval_precision_macro': '0.723', 'eval_recall_macro': '0.7239', 'eval_f1_macro': '0.7227', 'eval_runtime': '4.989', 'eval_samples_per_second': '545.2', 'eval_steps_per_second': '34.08', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '473', 'train_samples_per_second': '67.07', 'train_steps_per_second': '4.196', 'train_loss': '0.7438', 'epoch': '5'}
Entrenamiento de RoBERTa completado.


In [63]:
print("Evaluando RoBERTa en conjunto de prueba...")
roberta_preds = trainer_roberta.predict(test_dataset_roberta)
roberta_logits = roberta_preds.predictions
roberta_preds_ids = np.argmax(roberta_logits, axis=-1)

# Métricas globales
roberta_metrics = {
    'accuracy': accuracy_score(test_labels_ids, roberta_preds_ids),
    'precision_macro': precision_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0),
    'recall_macro': recall_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0),
    'f1_macro': f1_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0)
}

print(f"\nResultados RoBERTa - Test:")
print(f"   Accuracy: {roberta_metrics['accuracy']:.4f}")
print(f"   Precision (macro): {roberta_metrics['precision_macro']:.4f}")
print(f"   Recall (macro): {roberta_metrics['recall_macro']:.4f}")
print(f"   F1-Score (macro): {roberta_metrics['f1_macro']:.4f}")

# Matriz de confusión
roberta_cm_path = generar_matriz_confusion(
    test_labels_ids, roberta_preds_ids, clases,
    'RoBERTa', 'cm_roberta.png', RESULTADOS_DIR
)
print(f"Matriz de confusión guardada: {roberta_cm_path}")

# Reporte por clase
roberta_report = reporte_por_clase(test_labels_ids, roberta_preds_ids, clases)

# Curva ROC
roberta_probas = torch.nn.functional.softmax(torch.tensor(roberta_logits), dim=-1).numpy()
roberta_roc_path, roberta_auc_dict, roberta_macro_auc = generar_curva_roc(
    test_labels_ids, roberta_probas, clases,
    'RoBERTa', 'roc_roberta.png', RESULTADOS_DIR
)
print(f"Curva ROC guardada: {roberta_roc_path}")

print(f"\nAUC por clase:")
for i, clase in enumerate(clases):
    print(f"   {clase}: {roberta_auc_dict[i]:.4f}")
print(f"   Macro-promedio: {roberta_macro_auc:.4f}")

Evaluando RoBERTa en conjunto de prueba...

Resultados RoBERTa - Test:
   Accuracy: 0.7243
   Precision (macro): 0.7248
   Recall (macro): 0.7244
   F1-Score (macro): 0.7238
Matriz de confusión guardada: /content/resultados_transformers/cm_roberta.png
Curva ROC guardada: /content/resultados_transformers/roc_roberta.png

AUC por clase:
   Alegria: 0.8801
   Miedo: 0.9413
   Sorpresa: 0.8987
   Tristeza: 0.9059
   Macro-promedio: 0.9067



### 7. Exportación del reporte

In [64]:
cm_beto_rel = beto_cm_path.replace('\\', '/')
cm_roberta_rel = roberta_cm_path.replace('\\', '/')
roc_beto_rel = beto_roc_path.replace('\\', '/')
roc_roberta_rel = roberta_roc_path.replace('\\', '/')

ruta_reporte = os.path.join(RESULTADOS_DIR, 'reporte_transformers.md')

with open(ruta_reporte, 'w', encoding='utf-8') as f:
    f.write("# Reporte de evaluación de modelos Transformer\n\n")
    f.write("Resultados de métricas de rendimiento para los modelos BETO (BERT en Español) y RoBERTa (BERTIN - Español).\n\n")

    f.write("---\n\n")
    f.write("## Comparativa Global\n\n")
    f.write("| Métrica | BETO | RoBERTa |\n")
    f.write("|---------|------|---------|\n")
    f.write(f"| Accuracy | {beto_metrics['accuracy']:.4f} | {roberta_metrics['accuracy']:.4f} |\n")
    f.write(f"| Precision (macro) | {beto_metrics['precision_macro']:.4f} | {roberta_metrics['precision_macro']:.4f} |\n")
    f.write(f"| Recall (macro) | {beto_metrics['recall_macro']:.4f} | {roberta_metrics['recall_macro']:.4f} |\n")
    f.write(f"| F1-Score (macro) | {beto_metrics['f1_macro']:.4f} | {roberta_metrics['f1_macro']:.4f} |\n")
    f.write(f"| AUC (macro) | {beto_macro_auc:.4f} | {roberta_macro_auc:.4f} |\n\n")

    f.write("---\n\n")
    f.write("## BETO\n\n")
    f.write("### Métricas Globales\n\n")
    f.write(f"- **Accuracy:** {beto_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {beto_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {beto_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {beto_metrics['f1_macro']:.4f}\n\n")

    f.write("### AUC (Área Bajo la Curva ROC)\n\n")
    for i, clase in enumerate(clases):
        f.write(f"- **{clase}:** {beto_auc_dict[i]:.4f}\n")
    f.write(f"- **Macro-promedio:** {beto_macro_auc:.4f}\n\n")

    f.write("### Reporte por Clase\n\n")
    f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
    f.write("|-------|-----------|--------|----------|---------|\n")
    for clase in clases:
        r = beto_report[clase]
        f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
    f.write("\n")

    f.write("### Matriz de Confusión\n\n")
    f.write(f"![Matriz de Confusión BETO]({cm_beto_rel})\n\n")

    f.write("### Curva ROC\n\n")
    f.write(f"![Curva ROC BETO]({roc_beto_rel})\n\n")

    f.write("### Classification Report\n\n")
    f.write("```text\n")
    f.write(classification_report(test_labels_ids, beto_preds_ids,
                                  labels=range(len(clases)), target_names=clases, zero_division=0))
    f.write("```\n\n")

    f.write("---\n\n")
    f.write("## RoBERTa\n\n")
    f.write("### Métricas Globales\n\n")
    f.write(f"- **Accuracy:** {roberta_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {roberta_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {roberta_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {roberta_metrics['f1_macro']:.4f}\n\n")

    f.write("### AUC (Área Bajo la Curva ROC)\n\n")
    for i, clase in enumerate(clases):
        f.write(f"- **{clase}:** {roberta_auc_dict[i]:.4f}\n")
    f.write(f"- **Macro-promedio:** {roberta_macro_auc:.4f}\n\n")

    f.write("### Reporte por Clase\n\n")
    f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
    f.write("|-------|-----------|--------|----------|---------|\n")
    for clase in clases:
        r = roberta_report[clase]
        f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
    f.write("\n")

    f.write("### Matriz de Confusión\n\n")
    f.write(f"![Matriz de Confusión RoBERTa]({cm_roberta_rel})\n\n")

    f.write("### Curva ROC\n\n")
    f.write(f"![Curva ROC RoBERTa]({roc_roberta_rel})\n\n")

    f.write("### Classification Report\n\n")
    f.write("```text\n")
    f.write(classification_report(test_labels_ids, roberta_preds_ids,
                                  labels=range(len(clases)), target_names=clases, zero_division=0))
    f.write("```\n\n")

print(f"\nReporte guardado: {ruta_reporte}")
print("\nPROCESO COMPLETADO")
print("Se generaron:")
print(f"  - {os.path.basename(beto_cm_path)}")
print(f"  - {os.path.basename(beto_roc_path)}")
print(f"  - {os.path.basename(roberta_cm_path)}")
print(f"  - {os.path.basename(roberta_roc_path)}")
print(f"  - reporte_transformers.md")



Reporte guardado: /content/resultados_transformers/reporte_transformers.md

PROCESO COMPLETADO
Se generaron:
  - cm_beto.png
  - roc_beto.png
  - cm_roberta.png
  - roc_roberta.png
  - reporte_transformers.md
